# Cuaderno 3: Visualización y validación

Este cuaderno analiza y visualiza los resultados de la simulación del tsunami de 1979
en Tumaco. Incluye:

1. Mapa de inundación máxima
2. Series de tiempo en estaciones de observación históricas
3. Validación cuantitativa contra run-up observado (Herd et al., 1981)
4. Animación de la propagación del tsunami

**Prerrequisito:** Ejecutar `02-simulation.ipynb` para generar los archivos `h-*.np`.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q rasterio pyproj

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm, BoundaryNorm
from matplotlib.patches import FancyArrowPatch
from pathlib import Path

import rasterio
from pyproj import Transformer

# Directorios
WORK_DIR   = Path('/content') if IN_COLAB else Path('.')
OUTPUT_DIR = WORK_DIR / 'output_1979'
DATA_DIR   = Path('../data') if not IN_COLAB else WORK_DIR / 'data'
DEM_PATH   = WORK_DIR / 'tumaco_dem_utm_sim.tif'

print(f"Directorio de resultados: {OUTPUT_DIR}")

## 1. Cargar datos de la simulación

In [ ]:
# Cargar DEM
with rasterio.open(str(DEM_PATH)) as src:
    dem = src.read(1)
    dem_transform = src.transform
    dem_crs = src.crs
    RESOLUTION = abs(src.transform.a)  # metros/pixel
    NROWS, NCOLS = src.height, src.width

print(f"DEM: {NROWS}×{NCOLS} px | Resolución: {RESOLUTION:.0f} m/px")

# Cargar snapshots de la simulación
output_files = sorted(glob.glob(str(OUTPUT_DIR / 'h-*.np')))

if not output_files:
    print("⚠ No se encontraron archivos de simulación en:", OUTPUT_DIR)
    print("  Generando datos de ejemplo para demostración...")
    DEMO_MODE = True
else:
    DEMO_MODE = False
    print(f"✓ {len(output_files)} snapshots cargados")

# Construir diccionario tiempo → array
heightmaps = {}
if not DEMO_MODE:
    for fpath in output_files:
        t = float(Path(fpath).stem.split('-')[1])
        heightmaps[t] = np.load(fpath)
    t_range = sorted(heightmaps.keys())
    print(f"  Tiempo: {t_range[0]:.0f}s – {t_range[-1]:.0f}s")
else:
    # Modo demo: simular propagación simplificada para visualización
    t_range = list(range(0, 1860, 60))
    Ly = RESOLUTION * NCOLS
    for t in t_range:
        # Aproximación analítica simple de una ola gaussiana propagándose
        y_vec = np.linspace(0, Ly, NCOLS)
        c_wave = 200  # m/s velocidad media de la ola
        y_epi = Ly - 60_000
        y_wave = y_epi - c_wave * t  # propagación hacia la costa
        lamb = 80_000
        k1, k2 = 28.416 / lamb**2, 256. / lamb**2
        eta = 2 * (1.5 * np.exp(-k1 * (y_vec - (y_wave + 0.3153*lamb))**2)
                   - 0.5 * np.exp(-k2 * (y_vec - y_wave)**2))
        # Amplificación batimétrica simplificada (Green's law)
        depth = np.abs(np.minimum(dem, 0)).mean(axis=0).clip(1, None)
        amp = (depth.mean() / depth.clip(1))**0.25
        hmap = np.tile(eta * amp, (NROWS, 1)) - np.minimum(dem, 0)
        hmap = np.clip(hmap, 0.001, None)
        heightmaps[float(t)] = hmap.astype(np.float32)
    t_range = sorted(heightmaps.keys())

## 2. Mapa de inundación máxima

In [ ]:
# Calcular inundación máxima sobre tierra en toda la simulación
max_inundation = np.zeros_like(dem, dtype=np.float32)
max_water_surface = np.full_like(dem, -9999.0, dtype=np.float32)

for t, hmap in heightmaps.items():
    h_arr = hmap if hmap.shape == dem.shape else hmap[:dem.shape[0], :dem.shape[1]]
    # Altura del agua sobre el nivel del terreno (solo en tierra)
    land_mask = dem >= 0
    inund = np.where(land_mask, h_arr + dem, 0.0)
    max_inundation = np.maximum(max_inundation, inund)
    # Superficie del agua total
    max_water_surface = np.maximum(max_water_surface, h_arr + np.minimum(dem, 0))

print(f"Inundación máxima simulada: {max_inundation.max():.2f} m")
print(f"Área inundada estimada (inund > 0.1m): "
      f"{(max_inundation > 0.1).sum() * RESOLUTION**2 / 1e6:.1f} km²")

In [ ]:
# Coordenadas de lugares de interés en UTM 18N
transformer = Transformer.from_crs(4326, 32618, always_xy=True)

places = {
    'Tumaco': (1.82, -78.76),
    'Bocagrande': (1.73, -78.94),
    'Epicentro\n1979': (1.598, -79.359),
}
places_utm = {
    name: transformer.transform(lon, lat)
    for name, (lat, lon) in places.items()
}

# Convertir a índices de pixel
def utm_to_pixel(x_utm, y_utm, transform):
    col = int((x_utm - transform.c) / transform.a)
    row = int((y_utm - transform.f) / transform.e)
    return row, col

# Calcular extensión del DEM en km (para ejes del mapa)
x_origin = dem_transform.c / 1000
y_origin = dem_transform.f / 1000
x_end = (dem_transform.c + dem_transform.a * NCOLS) / 1000
y_end = (dem_transform.f + dem_transform.e * NROWS) / 1000

# Mapa de inundación máxima
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Panel izquierdo: inundación máxima ---
ax = axes[0]
# Fondo: DEM batimétrico
dem_bg = np.where(dem < 0, dem, np.nan)
ax.imshow(dem_bg, cmap='Blues', alpha=0.4,
          extent=[x_origin, x_end, y_end, y_origin], vmin=-5000, vmax=0)

# Capa de inundación
inund_masked = np.ma.masked_where(max_inundation <= 0.1, max_inundation)
im = ax.imshow(inund_masked, cmap='YlOrRd', vmin=0.1, vmax=6,
               extent=[x_origin, x_end, y_end, y_origin], alpha=0.85)
plt.colorbar(im, ax=ax, label='Inundación máxima (m)', shrink=0.8)

# Contorno de tierra
ax.contour(dem, levels=[0], colors='k', linewidths=0.8,
           extent=[x_origin, x_end, y_end, y_origin])

# Lugares de interés
for name, (x_utm, y_utm) in places_utm.items():
    is_epi = 'Epicentro' in name
    ax.plot(x_utm/1000, y_utm/1000,
            'r*' if is_epi else 'ko',
            markersize=12 if is_epi else 7, zorder=5)
    ax.annotate(name, (x_utm/1000, y_utm/1000),
                textcoords='offset points', xytext=(5, 5),
                fontsize=8, color='darkred' if is_epi else 'black')

ax.set_xlabel('Easting UTM 18N (km)')
ax.set_ylabel('Northing UTM 18N (km)')
ax.set_title('Inundación máxima — Tsunami Tumaco 1979\n(30 minutos de simulación)')

# --- Panel derecho: run-up en la línea de costa ---
ax2 = axes[1]
coast_mask = (dem >= -10) & (dem <= 5)
coast_runup = max_inundation.copy()
coast_runup[~coast_mask] = np.nan
im2 = ax2.imshow(coast_runup, cmap='hot_r', vmin=0, vmax=8,
                 extent=[x_origin, x_end, y_end, y_origin])
plt.colorbar(im2, ax=ax2, label='Run-up en zona costera (m)', shrink=0.8)
ax2.set_xlabel('Easting UTM 18N (km)')
ax2.set_title('Run-up en zona costera (<5 m s.n.m.)')

for name, (x_utm, y_utm) in places_utm.items():
    if 'Epicentro' not in name:
        ax2.plot(x_utm/1000, y_utm/1000, 'ko', markersize=7)
        ax2.annotate(name, (x_utm/1000, y_utm/1000),
                     textcoords='offset points', xytext=(5, 3), fontsize=8)

plt.tight_layout()
plt.savefig(WORK_DIR / 'max_inundation_map.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"Mapa guardado en: {WORK_DIR}/max_inundation_map.png")

## 3. Series de tiempo en estaciones de observación

Extraemos la serie temporal de altura del agua en las estaciones donde Herd et al.
(1981) reportaron observaciones de run-up del tsunami de 1979.

In [ ]:
# Estaciones de observación con coordenadas y run-up histórico
stations = [
    {'name': 'Tumaco (muelle)', 'lat': 1.82,  'lon': -78.76, 'obs_runup': 5.0,
     'obs_arrival': 15, 'color': 'firebrick'},
    {'name': 'Bocagrande',       'lat': 1.73,  'lon': -78.94, 'obs_runup': 3.0,
     'obs_arrival': 18, 'color': 'darkorange'},
    {'name': 'Esmeraldas (ECU)', 'lat': 0.97,  'lon': -79.65, 'obs_runup': 4.0,
     'obs_arrival': 12, 'color': 'steelblue'},
    {'name': 'El Charco',        'lat': 2.47,  'lon': -78.11, 'obs_runup': 1.5,
     'obs_arrival': 30, 'color': 'darkgreen'},
]

# Extraer series de tiempo de los heightmaps
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
fig.suptitle('Series de tiempo — Altura de la superficie del agua\nTsunami Tumaco 1979',
             fontsize=13)

for idx, (sta, ax) in enumerate(zip(stations, axes.flatten())):
    # Convertir coordenadas a píxel
    x_utm, y_utm = transformer.transform(sta['lon'], sta['lat'])
    try:
        row, col = utm_to_pixel(x_utm, y_utm, dem_transform)
        row = np.clip(row, 0, NROWS - 1)
        col = np.clip(col, 0, NCOLS - 1)
        elev = dem[row, col]

        # Extraer serie de tiempo
        times = np.array(sorted(heightmaps.keys()))
        h_vals = np.array([heightmaps[t][row, col] + min(dem[row, col], 0)
                           for t in times])
    except (IndexError, KeyError):
        # Si la estación está fuera del dominio, mostrar señal vacía
        times = np.array(sorted(heightmaps.keys()))
        h_vals = np.zeros_like(times)
        elev = 0.0

    # Graficar
    ax.plot(times / 60, h_vals, color=sta['color'], linewidth=2, label='Simulado')
    ax.axhline(0, color='k', linewidth=0.5, linestyle=':')
    ax.axvline(sta['obs_arrival'], color='gray', linewidth=1, linestyle='--',
               label=f"Arribo obs. ({sta['obs_arrival']} min)")
    ax.axhline(sta['obs_runup'], color=sta['color'], linewidth=1.5, linestyle='--',
               alpha=0.6, label=f"Run-up obs.: {sta['obs_runup']} m")

    ax.fill_between(times / 60, h_vals, 0,
                    where=h_vals > 0, alpha=0.2, color=sta['color'])
    ax.set_title(f"{sta['name']} (z={elev:.1f}m)", fontsize=10)
    ax.set_ylabel('Η (m)', fontsize=9)
    ax.set_ylim(-3, 8)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

for ax in axes[1]:
    ax.set_xlabel('Tiempo desde el sismo (minutos)')

plt.tight_layout()
plt.savefig(WORK_DIR / 'time_series.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Series de tiempo guardadas en: {WORK_DIR}/time_series.png")

## 4. Validación cuantitativa

Comparación del run-up máximo simulado vs. observado en 8 estaciones costeras
de Herd et al. (1981) y la base de datos NOAA NGDC.

In [ ]:
# Cargar observaciones históricas
try:
    obs_path = DATA_DIR / 'observaciones_1979.csv'
    df_obs = pd.read_csv(obs_path)
    print(f"Observaciones cargadas: {len(df_obs)} estaciones")
except FileNotFoundError:
    # Datos incrustados como fallback
    df_obs = pd.DataFrame({
        'estacion': ['Tumaco (muelle)', 'Tumaco (playa)', 'Bocagrande',
                     'San Juan', 'El Charco', 'Esmeraldas (ECU)',
                     'Buenaventura', 'Salinas (ECU)'],
        'lat': [1.82, 1.80, 1.73, 1.23, 2.47, 0.97, 3.88, -2.20],
        'lon': [-78.76, -78.80, -78.94, -77.97, -78.11, -79.65, -77.02, -80.97],
        'runup_m': [5.0, 4.5, 3.0, 2.5, 1.5, 4.0, 0.8, 1.2],
    })

# Calcular run-up simulado en cada estación
sim_runup = []
for _, row in df_obs.iterrows():
    try:
        x_utm, y_utm = transformer.transform(row['lon'], row['lat'])
        r, c = utm_to_pixel(x_utm, y_utm, dem_transform)
        r = np.clip(r, 0, NROWS - 1)
        c = np.clip(c, 0, NCOLS - 1)
        # Área de búsqueda de 3×3 píxeles alrededor del punto
        r0, r1 = max(0, r-2), min(NROWS, r+3)
        c0, c1 = max(0, c-2), min(NCOLS, c+3)
        max_val = max_inundation[r0:r1, c0:c1].max()
    except Exception:
        max_val = np.nan
    sim_runup.append(max_val)

df_obs['sim_runup_m'] = sim_runup
df_val = df_obs.dropna(subset=['sim_runup_m'])

# Métricas de validación
if len(df_val) > 0:
    rmse = np.sqrt(((df_val['runup_m'] - df_val['sim_runup_m'])**2).mean())
    bias = (df_val['sim_runup_m'] - df_val['runup_m']).mean()
    r_sq = np.corrcoef(df_val['runup_m'], df_val['sim_runup_m'])[0,1]**2
    print(f"Métricas de validación (n={len(df_val)})")
    print(f"  RMSE: {rmse:.2f} m")
    print(f"  Sesgo: {bias:+.2f} m")
    print(f"  R²: {r_sq:.3f}")

# Tabla comparativa
print("\nComparación por estación:")
print(df_obs[['estacion', 'runup_m', 'sim_runup_m']].to_string(index=False))

In [ ]:
# Gráfica de validación: observado vs. simulado
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: scatter observado vs. simulado
ax = axes[0]
v_max = max(df_obs['runup_m'].max(), df_obs['sim_runup_m'].fillna(0).max()) * 1.15
ax.plot([0, v_max], [0, v_max], 'k--', linewidth=1, label='1:1')
ax.plot([0, v_max], [0, 2*v_max], 'gray', linewidth=0.5, linestyle=':', alpha=0.5)
ax.plot([0, v_max], [0, v_max/2], 'gray', linewidth=0.5, linestyle=':', alpha=0.5)

sc = ax.scatter(df_obs['runup_m'], df_obs['sim_runup_m'],
                c=np.abs(df_obs['lat']), cmap='viridis', s=80, zorder=5)
plt.colorbar(sc, ax=ax, label='|Latitud| (°N)')

for _, row in df_obs.iterrows():
    ax.annotate(row['estacion'], (row['runup_m'], row['sim_runup_m'] or 0),
                fontsize=7, textcoords='offset points', xytext=(4, 3))

ax.set_xlim(0, v_max)
ax.set_ylim(0, v_max)
ax.set_xlabel('Run-up observado (m) — Herd et al. 1981')
ax.set_ylabel('Run-up simulado (m)')
ax.set_title('Validación del modelo\nTsunami Tumaco 1979')
ax.grid(True, alpha=0.3)

if len(df_val) > 0:
    ax.text(0.05, 0.92, f'RMSE = {rmse:.2f} m\nR² = {r_sq:.2f}\nSesgo = {bias:+.2f} m',
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Panel derecho: barras comparativas
ax2 = axes[1]
x_pos = np.arange(len(df_obs))
width = 0.38
ax2.bar(x_pos - width/2, df_obs['runup_m'], width,
        label='Observado (Herd et al. 1981)', color='steelblue', alpha=0.85)
ax2.bar(x_pos + width/2, df_obs['sim_runup_m'].fillna(0), width,
        label='Simulado (N-wave Mw 8.2)', color='firebrick', alpha=0.85)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(df_obs['estacion'], rotation=35, ha='right', fontsize=8)
ax2.set_ylabel('Run-up (m)')
ax2.set_title('Comparación run-up por estación')
ax2.legend(fontsize=9)
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(WORK_DIR / 'validation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Gráfica de validación guardada en: {WORK_DIR}/validation.png")

## 5. Animación de la propagación

Genera un GIF animado mostrando la propagación del tsunami desde el epicentro
hasta la costa de Tumaco durante los primeros 30 minutos.

In [ ]:
from matplotlib.animation import FuncAnimation

fig, ax = plt.subplots(figsize=(10, 7))

# Fondo fijo: terreno (hillshade)
from matplotlib.colors import LightSource
ls = LightSource(azdeg=315, altdeg=35)
dem_vis = np.where(dem > 0, dem, 0).astype(float)
hillshade = ls.hillshade(dem_vis, vert_exag=0.001)
ax.imshow(hillshade, cmap='gray', alpha=0.5,
          extent=[x_origin, x_end, y_end, y_origin])

# Capa de batimetría (océano)
ocean_mask = np.ma.masked_where(dem >= 0, dem)
ax.imshow(ocean_mask, cmap='Blues', vmin=-5000, vmax=0, alpha=0.4,
          extent=[x_origin, x_end, y_end, y_origin])

# Contorno de costa
ax.contour(dem, levels=[0], colors='k', linewidths=0.8,
           extent=[x_origin, x_end, y_end, y_origin])

# Epicentro
epi_x_utm, epi_y_utm = transformer.transform(-79.359, 1.598)
ax.plot(epi_x_utm/1000, epi_y_utm/1000, 'r*', markersize=14, zorder=10,
        label='Epicentro 1979')

# Capa animada de altura del agua
t_sorted = sorted(heightmaps.keys())
first_hmap = heightmaps[t_sorted[0]]
water_layer = ax.imshow(
    np.where(first_hmap > 0.5, first_hmap, np.nan),
    cmap='RdBu_r', vmin=-3, vmax=3, alpha=0.75,
    extent=[x_origin, x_end, y_end, y_origin]
)
plt.colorbar(water_layer, ax=ax, label='Elevación superficie (m)', shrink=0.7)

title_obj = ax.set_title('t = 0 s (0 min)', fontsize=12)
ax.set_xlabel('Easting UTM 18N (km)')
ax.set_ylabel('Northing UTM 18N (km)')
ax.legend(fontsize=9, loc='upper right')

def update_frame(frame_idx):
    t = t_sorted[frame_idx]
    hmap = heightmaps[t]
    # Mostrar solo perturbaciones significativas (> 0.1m o < -0.1m)
    water = np.where(np.abs(hmap + np.minimum(dem, 0)) > 0.1,
                     hmap + np.minimum(dem, 0), np.nan)
    water_layer.set_array(water)
    title_obj.set_text(f't = {t:.0f} s ({t/60:.0f} min)\nSimulación Tsunami Tumaco 1979')
    return [water_layer, title_obj]

# Crear animación (un frame por snapshot, 4 frames/segundo)
anim = FuncAnimation(
    fig, update_frame,
    frames=len(t_sorted),
    interval=250,    # ms entre frames
    blit=True
)

plt.tight_layout()

# Guardar como GIF
gif_path = WORK_DIR / 'tsunami_propagation.gif'
anim.save(str(gif_path), writer='pillow', fps=4, dpi=100)
print(f"Animación guardada: {gif_path}")
plt.close()

In [ ]:
# Mostrar la animación en Colab
from IPython.display import Image, display
if gif_path.exists():
    display(Image(str(gif_path)))

## 6. Resumen de resultados

In [ ]:
print("=" * 60)
print("RESUMEN DE RESULTADOS — SIMULACIÓN TSUNAMI TUMACO 1979")
print("=" * 60)
print(f"  Modelo:            tsunamiTPUlab v1.0 (Smarras et al. 2023)")
print(f"  Evento:            12-dic-1979, Mw 8.2")
print(f"  Resolución DEM:    {RESOLUTION:.0f} m (GEBCO 2023 / ETOPO 2022)")
print(f"  Duración sim.:     {max(t_sorted)/60:.0f} minutos")
print(f"  Snapshots:         {len(t_sorted)}")
print()
print("  --- Resultados ---")
print(f"  Inundación máx.:   {max_inundation.max():.2f} m")
inund_area = (max_inundation > 0.1).sum() * RESOLUTION**2 / 1e6
print(f"  Área inundada:     {inund_area:.2f} km²")
if len(df_val) > 0:
    print(f"  RMSE validación:   {rmse:.2f} m")
    print(f"  R²:                {r_sq:.3f}")
    print(f"  Sesgo:             {bias:+.2f} m")
print()
print("  --- Archivos generados ---")
for fname in ['max_inundation_map.png', 'time_series.png',
              'validation.png', 'tsunami_propagation.gif']:
    fp = WORK_DIR / fname
    if fp.exists():
        print(f"  ✓ {fname} ({fp.stat().st_size/1e3:.0f} KB)")
    else:
        print(f"  ✗ {fname} (no generado)")